---  
 
## ***Before Beginning***

- Yesterday, you directly wrote the code w = w - lr * w.grad, right? optimizer.step() is like a magical line that does that for you.

#### ***Training Loop***
- The structure of the training loop is used almost as is in the CNN and RNN models you will learn in the future.
- The following pseudo-code is the learning form of most artificial intelligence.

In [ ]:
# 1. Define model, loss function, optimizer
model = MyModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 2. Prepare DataLoader
dataloader = DataLoader(...)

# 3. Training Loop (Repeat N epochs)
for epoch in range(num_epochs):
    # Get mini-batch from DataLoader
    for data, labels in dataloader:
        # 3-1. Initialize Gradients
        optimizer.zero_grad()

        # 3-2. Forward Pass
        outputs = model(data)

        # 3-3. Calculate Loss
        loss = criterion(outputs, labels)

        # 3-4. Backward Pass
        loss.backward()

        # 3-5. Update Parameters
        optimizer.step()

    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

#### ***Common Mistakes***

- Missing optimizer.zero_grad(): "Why do I need to zero the gradients every time?"
- Because PyTorch 'accumulates' gradients instead of 'overwriting' them when calling backward()

- Tensor Device Mismatch (CPU vs GPU):
    * Errors occurring because the model is on the GPU but the data remains on the CPU are very common.
    * You must send the data to the same device as the model within the training loop, like data = data.to(device), labels = labels.to(device).

- Switching the Model's Evaluation Mode:
    * It's not essential, but a brief introduction to the concepts of model.train() and model.eval()
    * It's not very important now, but from Day 5 onwards, there are cases where the model's behavior differs between training and evaluation.
    * It's good to get into the habit of switching between these two modes from now on.  
  
  
---


## ***Day 4 Advanced Supplementary Learning:***
### Mastering Building a Real Neural Network (Revised Edition)

#### Learning Objective: To gain the ability to build a complete training pipeline in a reusable structure by automating repetitive tasks using PyTorch's high-level APIs (nn.Module, Optimizer, DataLoader).

### ***Concept Check Quiz***
This quiz will thoroughly check your understanding of the core concepts.

1. What are the respective roles of the __init__ and forward methods in an nn.Module class?  
    [Answer](Day05_quiz_1.ipynb)

2. What is the role of an Optimizer, and at which stage of the training loop is it used?  
    [Answer](Day05_quiz_2.ipynb)  
    ​
3. What are the respective roles of Dataset and DataLoader, and why should they be used together?

4. List the three core steps of the training loop (optimizer.zero_grad(), loss.backward(), optimizer.step()) in the correct order and explain the role of each step.

5. What does the code nn.Linear(in_features=10, out_features=5) mean? What are the input and output sizes of this layer?

6. What is the term for one full pass through the entire dataset during training? And what does the batch_size in a DataLoader signify?

7. Why do we pass model.parameters() to the optimizer? What would happen if this code were omitted?

8. Name one loss function typically used for regression problems and one for classification problems in PyTorch, and briefly explain their difference.

9. What is the purpose of the loss.item() code? What is the difference if you print the loss tensor itself without .item()?

10. Why do we use model.train() mode for training and model.eval() mode for evaluation? (Hint: Dropout, BatchNorm, etc.)

11. In the code torch.arange(100).view(-1, 1), what role does .view(-1, 1) play?

12. What is the main reason for using activation functions like nn.ReLU in a neural network model? What limitations would a model have without activation functions?

13. While training, the loss value does not decrease at all or even diverges. What hyperparameter should be suspected first, and how should it be adjusted?

14. What are the conceptual differences between the SGD and Adam optimizers? Which one generally tends to converge faster?

15. What should the __getitem__ method of a CustomDataset class return, and in what data type?

16. When loss.backward() is called, what value is stored in the .grad attribute of a tensor that was set with requires_grad=False?

### ***Code Walkthroughs and Exercises***
  

Increase your adaptability to the PyTorch pipeline structure by following code examples for various scenarios and immediately solving related exercises.

- **Topic 1: Basic Linear Regression (Review)**
    - Scenario: Create the most basic regression model that predicts a single output (y) from a single input (x).
    - Core Concepts: nn.Linear(1, 1), nn.MSELoss

In [ ]:
# [Topic 1] Core Code
# 0. Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# 1. Prepare data: y = 3x + 5
X1 = torch.arange(1, 101, 1, dtype=torch.float32).view(-1, 1)
y1 = 3 * X1 + 5 + torch.randn(100, 1) * 2

# 2. Define Dataset and DataLoader
class CustomDataset(Dataset):
    def __init__(self, x, y): self.x, self.y = x, y
    def __getitem__(self, i): return (self.x[i], self.y[i])
    def __len__(self): return len(self.x)

train_dataset1 = CustomDataset(X1, y1)
train_loader1 = DataLoader(train_dataset1, batch_size=10, shuffle=True)

# 3. Define model, loss function, and optimizer
model1 = nn.Linear(in_features=1, out_features=1)
criterion1 = nn.MSELoss()
optimizer1 = torch.optim.SGD(model1.parameters(), lr=0.01)

# 4. Training loop
print("--- Starting Basic Linear Regression Training ---")
for epoch in range(50):
    for inputs, labels in train_loader1:
        outputs = model1(inputs)
        loss = criterion1(outputs, labels)
        optimizer1.zero_grad()
        loss.backward()
        optimizer1.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/50], Loss: {loss.item():.4f}')

--- Starting Basic Linear Regression Training ---
Epoch [10/50], Loss: nan
Epoch [20/50], Loss: nan
Epoch [30/50], Loss: nan
Epoch [40/50], Loss: nan
Epoch [50/50], Loss: nan


- **[Topic 1] Programming Exercises**  
    - **Problem 1-1 (Optimizer Swap & Comparison)**: 
        - Change the optimizer in the code above from torch.optim.SGD to torch.optim.Adam and retrain the model. With the same learning rate and number of epochs, which optimizer, SGD or Adam, reduces the loss faster?
    - **Problem 1-2 (Visualize Training Process)**: 
        - Store the calculated loss value for each epoch in a Python list. After training is complete, use Matplotlib to plot a line graph showing the trend of loss reduction against the epochs.  
    ---

- **Topic 2: Multiple Linear Regression**
    - Scenario: Create a model that predicts a single output (y) from two inputs (x1, x2).
    - Core Concept: nn.Linear(2, 1). This handles the situation where the number of input features increases from one to two.


In [ ]:
# [Topic 2] Core Code
# 1. Prepare data: y = 2*x1 + 3*x2 + 4
X2_1 = torch.randn(200, 1)
X2_2 = torch.randn(200, 1)
X2 = torch.cat([X2_1, X2_2], dim=1) # Combine the two input features
y2 = 2 * X2_1 + 3 * X2_2 + 4 + torch.randn(200, 1) * 2

# 2. Define Dataset and DataLoader (reusing CustomDataset)
train_dataset2 = CustomDataset(X2, y2)
train_loader2 = DataLoader(train_dataset2, batch_size=10, shuffle=True)

# 3. Define model, loss function, and optimizer
# Since there are 2 input features, set in_features=2
model2 = nn.Linear(in_features=2, out_features=1)
criterion2 = nn.MSELoss()
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.01)

# 4. Training loop
print("\n--- Starting Multiple Linear Regression Training ---")
for epoch in range(100):
    for inputs, labels in train_loader2:
        outputs = model2(inputs)
        loss = criterion2(outputs, labels)
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')



--- Starting Multiple Linear Regression Training ---
Epoch [10/100], Loss: 3.5449
Epoch [20/100], Loss: 4.8587
Epoch [30/100], Loss: 1.1631
Epoch [40/100], Loss: 5.2069
Epoch [50/100], Loss: 6.3216
Epoch [60/100], Loss: 2.5471
Epoch [70/100], Loss: 4.7388
Epoch [80/100], Loss: 4.0300
Epoch [90/100], Loss: 2.6805
Epoch [100/100], Loss: 5.8750


- **[Topic 2] Programming Exercises**
    - **Problem 2-1 (Model Architecture Change)**: 
        - The current model is a single linear layer. Create a deeper model by adding a hidden layer, like nn.Linear(in_features=2, out_features=10) -> nn.ReLU() -> nn.Linear(in_features=10, out_features=1). Confirm that the training loop code remains almost the same even when the model's architecture changes.

    - **Problem 2-2 (Learning Rate Tuning)**: 
        - In the optimizer from the code above, change the learning rate (lr) to 10 times (0.1) and 0.1 times (0.001) its current value and run the training again. Observe how the loss value changes (e.g., diverges or decreases too slowly) when the learning rate is too high or too low, and think about the reason.  
---

- **Topic 3: Simple Binary Classification**
    - **Scenario**: Create a model that classifies a score (x) as pass (1) if it's 50 or above, and fail (0) otherwise.
    - **Core Concepts**: nn.Sigmoid activation function, nn.BCELoss (Binary Cross Entropy Loss)

In [ ]:
# [Topic 3] Core Code
# 1. Prepare data: Pass/Fail based on a score of 50
X3 = torch.arange(0, 100, 1, dtype=torch.float32).view(-1, 1)
# 1 (Pass) if score is >= 50, otherwise 0 (Fail)
y3 = (X3 >= 50).float()

# 2. Define Dataset and DataLoader (reusing CustomDataset)
train_dataset3 = CustomDataset(X3, y3)
train_loader3 = DataLoader(train_dataset3, batch_size=10, shuffle=True)

# 3. Define model, loss function, and optimizer
class BinaryClassifier(nn.Module):
    def __init__(self):
        super(BinaryClassifier, self).__init__()
        self.linear = nn.Linear(1, 1)
        # Sigmoid function to convert output to a probability between 0 and 1
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.linear(x))

model3 = BinaryClassifier()
# For binary classification problems, use BCELoss
criterion3 = nn.BCELoss()
optimizer3 = torch.optim.Adam(model3.parameters(), lr=0.1)

# 4. Training loop
print("\n--- Starting Binary Classification Training ---")
for epoch in range(100):
    for inputs, labels in train_loader3:
        outputs = model3(inputs)
        loss = criterion3(outputs, labels)
        optimizer3.zero_grad()
        loss.backward()
        optimizer3.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')

# 5. Check results after training
print("\n--- Testing Classification Results ---")
model3.eval() # Switch to evaluation mode
with torch.no_grad(): # Disable gradient calculation
    test_scores = torch.tensor([[10.0], [49.0], [51.0], [95.0]])
    predictions = model3(test_scores)
    # If predicted probability is >= 0.5, it's a Pass (True)
    results = predictions >= 0.5
    for score, result in zip(test_scores, results):
        print(f"Score: {score.item():.0f} -> Pass: {result.item()}")



--- Starting Binary Classification Training ---
Epoch [10/100], Loss: 0.3359
Epoch [20/100], Loss: 0.0986
Epoch [30/100], Loss: 0.1626
Epoch [40/100], Loss: 0.2024
Epoch [50/100], Loss: 0.0382
Epoch [60/100], Loss: 0.1122
Epoch [70/100], Loss: 0.0214
Epoch [80/100], Loss: 0.0245
Epoch [90/100], Loss: 0.0515
Epoch [100/100], Loss: 0.1978

--- Testing Classification Results ---
Score: 10 -> Pass: False
Score: 49 -> Pass: True
Score: 51 -> Pass: True
Score: 95 -> Pass: True


- **[Topic 3] Programming Exercises**
    - **Problem 3-1 (Calculate Accuracy)**: 
        - After training is complete, feed the entire training dataset (X3, y3) into the model to make predictions. Compare the predictions with the actual labels to calculate the model's accuracy. (Accuracy = Number of Correct Predictions / Total Number of Predictions)

    - **Problem 3-2 (Save and Load Model)**: 
        - Save the parameters of the trained BinaryClassifier model to a file using torch.save(model3.state_dict(), 'classifier.pth'). Then, create a new model object, load the saved parameters using model.load_state_dict(torch.load('classifier.pth')), and confirm that the model is in its trained state by re-running the "Testing Classification Results" code.  
--- 

### ***3. Mini-Projects***

Choose two of the following three topics and build a model from scratch using a real dataset.

#### ***Project A***: California Housing Price Prediction (Regression)
- Objective: 
    - Use the California housing dataset from Scikit-learn to create a regression model that predicts housing prices based on various housing-related features (median income, house age, etc.).

- Data: sklearn.datasets.fetch_california_housing

- Core Task: Build a multiple linear regression model that takes 8 input features to predict 1 housing price, and train it using MSELoss.

#### ***Project B***: Breast Cancer Diagnosis (Binary Classification)
- Objective: 
    - Use the breast cancer dataset from Scikit-learn to create a model that classifies tumors as malignant or benign based on various tumor characteristics.

- Data: sklearn.datasets.load_breast_cancer

- Core Task: Build a binary classification model that takes 30 input features to predict 1 classification result (0 or 1). You must use Sigmoid and BCELoss.

#### ***Project C***: Wine Type Classification (Multi-class Classification - Challenge!)
- Objective: 
    - Use the wine dataset from Scikit-learn to create a model that classifies wine into one of three types based on its chemical properties.

- Data: sklearn.datasets.load_wine

- Core Task: Build a model that takes 13 input features to classify into 3 classes. The out_features of the final output layer should be 3, and you should use nn.CrossEntropyLoss as the loss function. (Hint: In multi-class classification, Softmax is used instead of Sigmoid at the end, but nn.CrossEntropyLoss includes Softmax internally, so your model's forward method only needs to return the result of the final nn.Linear layer.)